# 11. Functional Programming
Exhaustive guide to lambdas sorting, loop scopes late bindings, map/filter/reduce pipelines, partial functions and itertools.

This notebook uses the shared Fintech dataset `data/raw_transactions.csv` for combined analysis questions at the end.

In [ ]:
# Setup: Locate the Shared Dataset
import os
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
print("Using CSV file path:", csv_path)

### 1. Lambda Inline Definitions
**Explanation**: Declares anonymous functions.

**Syntax**:
```python
lambda argument: expression
```



In [ ]:
multiplier_lambda = lambda value: value * 2
print(multiplier_lambda(5))

### 2. Lambdas in Sorting Comparisons
**Explanation**: Provide sorting keys using lambdas.

**Syntax**:
```python
list_object.sort(key=lambda item: item[index])
```



In [ ]:
pairs_list = [(1, 'b'), (2, 'a')]
pairs_list.sort(key=lambda item: item[1])
print(pairs_list)

### 3. Loop Late Binding Values Gotcha
**Explanation**: Lambdas capture scope variables dynamically at call time, which can trigger loop variable gotchas.

**Syntax**:
```python
lambdas_list = [lambda: loop_index for loop_index in range(3)]
```

**Visual Explanation (Data with Baraa Style)**:
```mermaid
graph TD
    loop[for loop executes] -->|loop_index ends at 2| end_state
    callable[lambda calls] -->|reads late bound loop_index| res(2, 2)
```


In [ ]:
trap_list = [lambda: loop_index for loop_index in [1, 2]]
print([lambda_func() for lambda_func in trap_list])

### 4. Loop Late Binding Default Fixes
**Explanation**: Resolve late bindings by capturing loop values as parameter defaults.

**Syntax**:
```python
lambda parameter_name=loop_value: parameter_name
```

**Visual Explanation (Data with Baraa Style)**:
```mermaid
graph TD
    lambda_def[lambda val=loop_index] -->|captures val at definition time| isolated_val(Captured value instance)
```


In [ ]:
safe_list = [lambda parameter_name=loop_index: parameter_name for loop_index in [1, 2]]
print([lambda_func() for lambda_func in safe_list])

### 5. Mapping Iterables (map)
**Explanation**: Transforms iterables lazily.

**Syntax**:
```python
map(function, iterable)
```



In [ ]:
payout_records = [1, 2]
print(list(map(lambda x: x**2, payout_records)))

### 6. Filtering Iterables (filter)
**Explanation**: Filters elements lazily based on conditions.

**Syntax**:
```python
filter(function_filter, iterable)
```



In [ ]:
payout_records = [1, 2, 3]
print(list(filter(lambda x: x%2==0, payout_records)))

### 7. Cumulative Reductions (reduce)
**Explanation**: Aggregates values cumulatively.

**Syntax**:
```python
from functools import reduce
reduce(function_accumulator, sequence, initializer)
```



In [ ]:
from functools import reduce
payout_records = [1, 2, 3]
print(reduce(lambda acc, val: acc+val, payout_records))

### 8. Map-Filter Pipeline Combinations
**Explanation**: Chain map and filter operations.

**Syntax**:
```python
map(func, filter(pred, sequence))
```



In [ ]:
payout_records = [1, 2, 3]
print(list(map(lambda x: x*2, filter(lambda x: x>2, payout_records))))

### 9. Lazy Evaluations Iterators
**Explanation**: Evaluates items only when consumed.

**Syntax**:
```python
map_object = map(func, sequence)
```



In [ ]:
lazy_iterator = map(lambda x: x, [1, 2])
print(type(lazy_iterator))

### 10. Any and All Checks
**Explanation**: Evaluates boolean containment.

**Syntax**:
```python
any(sequence)
all(sequence)
```



In [ ]:
print('Any True?:', any([False, True]))

### 11. Operator module helpers
**Explanation**: Uses operator getters to speed up extraction.

**Syntax**:
```python
from operator import itemgetter
itemgetter(index)
```



In [ ]:
from operator import itemgetter
pairs_list = [(1, 'b'), (2, 'a')]
pairs_list.sort(key=itemgetter(1))
print(pairs_list)

### 12. Functools partial function bindings
**Explanation**: Binds parameters to construct partial functions.

**Syntax**:
```python
from functools import partial
partial_function = partial(original_function, fixed_argument)
```



In [ ]:
from functools import partial
def add_values(a, b): return a + b
add_five = partial(add_values, 5)
print(add_five(10))

### 13. Itertools generators: accumulate
**Explanation**: Yields running cumulative calculations.

**Syntax**:
```python
from itertools import accumulate
accumulate(iterable)
```



In [ ]:
from itertools import accumulate
print(list(accumulate([1, 2, 3])))

### 14. Itertools generators: chain
**Explanation**: Chains multiple iterables sequentially.

**Syntax**:
```python
from itertools import chain
chain(iterable_one, iterable_two)
```



In [ ]:
from itertools import chain
print(list(chain([1], [2])))

### 15. Itertools generators: groupby
**Explanation**: Groups sorted iterables by key functions.

**Syntax**:
```python
from itertools import groupby
groupby(sequence, key_function)
```



In [ ]:
from itertools import groupby
for group_key, group_generator in groupby([('a', 1), ('a', 2)], lambda x: x[0]):
    print(group_key, list(group_generator))

## Section 3: Fintech Interview Questions

### Q1: Write a map-filter-reduce pipeline on the first 50 transactions to calculate the sum of transaction amounts for 'Failed' entries.

In [ ]:
# Solution:
from functools import reduce
rows = []
with open(csv_path, 'r') as f:
    f.readline()
    for _ in range(50):
        rows.append(f.readline().strip().split(','))
        
failed_txs = filter(lambda r: r[5] == 'Failed', rows)
amounts = map(lambda r: float(r[3]) if r[3] not in ('', 'NaN') else 0.0, failed_txs)
failed_sum = reduce(lambda acc, val: acc + val, amounts, 0.0)
print('Total Failed Sum:', failed_sum)


### Q2: Implement standard partial functions `usd_to_eur` binding conversion factor 0.92, mapping amounts from the first 5 rows.

In [ ]:
# Solution:
from functools import partial
def convert(val, rate): return round(val * rate, 2)
usd_to_eur = partial(convert, rate=0.92)

with open(csv_path, 'r') as f:
    f.readline()
    for _ in range(5):
        row = f.readline().strip().split(',')
        amt = float(row[3]) if row[3] not in ('', 'NaN') else 0.0
        print('USD:', amt, '-> EUR:', usd_to_eur(amt))
